Basic Definitions
Zero-shot inference
The model performs a task without being given any example beforehand.
It relies entirely on the instruction itself and its general knowledge to infer what is expected.

Example:

Prompt: "Translate this sentence into Spanish: 'Hello, how are you?'"
→ The model understands it needs to translate, even without a prior example.

One-shot inference
The model is given one example of the task before being asked to perform it on a new input.

Example:

Prompt:
Example: "Hello" → "Hola"
Now translate: "How are you?" →
→ The model uses the format of the example to guide its response.

Key Differences
Criterion	Zero-shot	One-shot
Example provided	None	One
Relies on general context	Yes	Yes, plus the provided example
Accuracy	Less accurate for complex tasks	Often more accurate
When to use	Common or well-known tasks	Ambiguous or structured tasks

Inference in Practice
Here, "inference" refers to the model’s ability to deduce or apply a logical rule based on what it's been shown.

Zero-shot inference: the model infers the task using only the instruction.

One-shot inference: the model infers the task by following the structure of a single example.

Quick Comparison
Zero-shot:

Prompt: "Give the antonym of: 'fast'"
→ Output: "slow"

One-shot:

Example: "big" → "small"
Now: "fast" →
→ Output: "slow"

Other Types of Inference
Few-shot
The model is given several examples before being asked to respond.
This usually improves accuracy because it can infer a clear pattern from the examples.

Example:

Examples:
"cat" → "chat"
"dog" → "chien"
"bird" → "oiseau"
Now: "horse" →
→ Output: "cheval"

Multi-shot inference
This term is sometimes used interchangeably with few-shot, though it can also refer to scenarios where the model sees multiple turns or multiple tasks (not just inputs) that are connected. It’s less formally defined but still related to few-shot learning.

Chain-of-thought inference
The model is encouraged to explain its reasoning step by step before giving the answer.
This is especially useful for complex tasks like math or logic.

Example:

Question: Marie has 3 apples and gives 1 to Paul. How many does she have left?
Response:
"Marie starts with 3 apples. She gives 1 to Paul. That leaves her with 2 apples."

### Instruction à suivre pour exécution sur Google Colab

Aller dans `Execution -> Modifier le type d'exécution` puis sélectionner `T4-GPU` pour exploiter les fonctionnalités GPU.

![Colab GPU](resources/colab_gpu.png "T4-GPU")

## Installation des dépendances

Installons les dépendances nécessaires :
- **transformers** : la bibliothèque permettant de mettre en oeuvre les LLM exploitant le modèle transformers
- **torch** : célèbre bibliothèque de deep learning, sous jacente à transformers

In [ ]:
%pip install -U datasets

%pip install --upgrade pip
%pip install --disable-pip-version-check \
    torch \
    torchdata

%pip install \
    transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

## Chargement du LLM

Pour interagir avec un LLM, nous allons d'abord devoir le télécharger. Pour cet exemple, nous choissons un modèle simple et "léger" : flan-t5.

Nous chargeons également le **Tokenizer** afin de convertir le texte en tokens et vice-versa.

In [ ]:
model_name='google/flan-t5-base'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

**Exercice** : en vous aidant de la documentation https://huggingface.co/docs/transformers/llm_tutorial :
- Générer et afficher les tokens (ids) de la phrase (encodage)
- Décoder la liste de tokens (ids) et afficher la phrase (décodage)

In [ ]:
sentence = "Que peux-tu me dire sur les LLMs ?"

# Encoder la phrase en tokens : suite d'ID ; 1 ID = 1 token

# METTRE ICI LE CODE POUR ENCODER UNE PHRASE EN TOKEN ET L'AFFICHER

# METTRE ICI LE CODE POUR DECOER UNE SEQUENCE D'ID EN PHRASE ET L'AFFICHER

## Interrogation du LLM

A présent, utilisons notre LLM pour générer du texte.

Notez la syntaxe `User: question? Assistant: "`. Nous utilisons cette syntaxe car le LLM est un modèle qui génère la suite de la phrase et cette syntaxe lui permet de comprendre ce qu'on attend de lui. Ceci à la différence des modèles d'instruction qui génèrent une réponse pour une instruction donnée.

In [ ]:
sentence = "User: quelle est la capitale de la France ?Assistant: "

inputs = tokenizer(sentence, return_tensors='pt') # return les tenseurs au format pytorch
output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=50
    )[0],
    skip_special_tokens=True # Ne pas retourner les tokens <s>, </s>, ...
)

print(output)

Paris


## Résumé de dialogues

Dans cette partie, nous allons utiliser le LLM pour résumer des dialogues.

Tout d'abord, téléchargeons le dataset [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) depuis Huggingface

In [ ]:
huggingface_dataset_name = "knkarthick/dialogsum"
dataset = load_dataset(huggingface_dataset_name)

Affichons 2 exemples de dialogues, les exemples numéro 40 et 200.

In [ ]:
example_indices = [40, 200]

dash_line = '-'.join('' for x in range(100))

for i, index in enumerate(example_indices):
    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print('DIALOGUE D ENTREE:')
    print(dataset['test'][index]['dialogue'])
    print(dash_line)
    print('RESUME HUMAIN:')
    print(dataset['test'][index]['summary'])
    print(dash_line)
    print()

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
DIALOGUE D ENTREE:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
---------------------------------------------------------------------------------------------------
RESUME HUMAIN:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------

---------------------------------------------------------------------------------------------------
Example  

Tentons une première approche pour résumer les dialogues 40 et 200.

In [ ]:
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']

    inputs = tokenizer(dialogue, return_tensors='pt') # retourner les tenseurs
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            max_new_tokens=50, # max 50 tokens générés
        )[0],
        skip_special_tokens=True # on ne génère pas les tokens spéciaux <, >, ...
    )

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'DIALOGUE D ENTREE::\n{dialogue}')
    print(dash_line)
    print(f'RESUME HUMAIN:\n{summary}')
    print(dash_line)
    print(f'RESUME PAR LLM SANS PROMPT ENGINEERING:\n{output}\n')

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
DIALOGUE D ENTREE::
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
---------------------------------------------------------------------------------------------------
RESUME HUMAIN:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------
RESUME PAR LLM SANS PROMPT ENGINEERING:
Person1: It's ten to nine.

-----------------------------------------

**Exercice** : selon vous est-ce que le résumé est bon ? Pourquoi ?

## Résumé avec un prompt Instruction

Dans l'exemple ci-dessous, ajoutons une instruction indiquant au LLM ce qu'il doit faire.

### 1. Zero shot inference

Pour amener le modèle à accomplir une tâche, comme résumer un dialogue, vous pouvez transformer ce dialogue en une consigne spécifique. Cela est connu sous le nom d'inférence zéro-shot.

En encadrant le dialogue dans une consigne descriptive, vous pourrez observer les modifications apportées au texte généré.

In [ ]:
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']

    prompt = f"""
Summarize the following conversation between two persons to extract the key points of the conversation.

{dialogue}

Summary:
    """

    # Input constructed prompt instead of the dialogue.
    inputs = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'DIALOGUE D ENTREE:\n{prompt}')
    print(dash_line)
    print(f'RESME HUMAIN:\n{summary}')
    print(dash_line)
    print(f'>>>RESME AVEC ZERO SHOT INFERENCE:\n{output}\n')

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
DIALOGUE D ENTREE:

Summarize the following conversation between two persons to extract the key points of the conversation.

#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

Summary:
    
---------------------------------------------------------------------------------------------------
RESME HUMAIN:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
------------------------------------------------------------------------------------------

C'est déjà mieux ! Mais on peut encore faire mieux. Essayons de rajouter un exemple de résumé.

### 2. One Shot Inference

L'inférence one-shot et few-shot consiste à fournir au modèle de langage un ou plusieurs exemples complets de paires consigne-réponse correspondant à votre tâche avant de lui soumettre la consigne réelle que vous souhaitez qu'il accomplisse. Cela s'appelle "l'apprentissage en contexte" (_in context learning_), et cela permet au modèle de comprendre votre tâche spécifique. Pour en savoir plus, vous pouvez consulter [cet article](https://huggingface.co/blog/few-shot-learning-gpt-neo-and-inference-api).

In [ ]:
def make_prompt(example_indices_full, example_index_to_summarize):
    prompt = ''
    for index in example_indices_full:
        dialogue = dataset['test'][index]['dialogue']
        summary = dataset['test'][index]['summary']

        # The stop sequence '{summary}\n\n\n' is important for FLAN-T5. Other models may have their own preferred stop sequence.
        prompt += f"""
Dialogue:

{dialogue}

Summary:
{summary}


"""

    dialogue = dataset['test'][example_index_to_summarize]['dialogue']

    prompt += f"""
Dialogue:

{dialogue}

Summary:
"""

    return prompt

Construsons le prompt et affichons le.

In [ ]:
example_indices_full = [40]
example_index_to_summarize = 200

one_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)
# one_shot_prompt is a string
print(one_shot_prompt)


Dialogue:

#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

Summary:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.



Dialogue:

#Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more 

Lançons l'inférence sur un dialogue, qui doit bien entendu être différent de celui utilisé pour réaliser l'exemple.

In [ ]:
summary = dataset['test'][example_index_to_summarize]['summary']

inputs = tokenizer(one_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'RESME HUMAIN:\n{summary}\n')
print(dash_line)
print(f'>>>RESME LLM AVEC ONE SHOT INFERENCE:\n{output}')

---------------------------------------------------------------------------------------------------
RESME HUMAIN:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

---------------------------------------------------------------------------------------------------
>>>RESME LLM AVEC ONE SHOT INFERENCE:
#Person1#: You could add a painting program to your software. #Person2#: That would be a bonus. #Person1#: You might also want to upgrade your hardware because it is pretty outdated now


## Influence des paramètres du LLM

Nous allons maintenant faire varier plusieurs paramètres du LLM :
- température
- top_k
- top_p
- sampling

Pour cela aidez-vous de la documentation :
- https://huggingface.co/docs/transformers/generation_strategies
- https://huggingface.co/docs/transformers/main_classes/text_generation